In [1]:
# -------------------------------------------------------------------------
# 0. Clean & Safe Dependencies Installation (No PyTorch overwriting)
# -------------------------------------------------------------------------
!pip install -q faiss-cpu transformers peft bitsandbytes accelerate timm evaluate rouge-score nltk scikit-learn seaborn matplotlib



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/18.8 MB ? eta -:--:--

   ━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/18.8 MB 32.3 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━ 12.6/18.8 MB 174.2 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 18.8/18.8 MB 186.6 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 78.5 MB/s eta 0:00:00


   ━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/43.1 MB 118.1 MB/s eta 0:00:01

... (5 outputs left out)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 43.1/43.1 MB 124.9 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 43.1/43.1 MB 124.9 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/84.1 kB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.7 MB/s eta 0:00:00


In [2]:
# -------------------------------------------------------------------------
# 1. Comprehensive Data Manifest & Cohort Extraction (p10/p11)
# -------------------------------------------------------------------------
import os, sys, json, subprocess, zipfile, pandas as pd, numpy as np
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit

work_dir = "/kaggle/working"
os.makedirs(work_dir, exist_ok=True)
os.chdir(work_dir)

PHYSIONET_USER = os.environ.get("PHYSIONET_USER", "")
PHYSIONET_PASS = os.environ.get("PHYSIONET_PASS", "")

netrc_content = "machine physionet.org\nlogin " + str(PHYSIONET_USER) + "\npassword " + str(PHYSIONET_PASS) + "\n"
netrc_path = os.path.expanduser("~/.netrc")
with open(netrc_path, "w", encoding="utf-8") as f:
    f.write(netrc_content)
os.chmod(netrc_path, 0o600)

metadata_files = [
    ("meta.csv", "https://physionet.org/files/mimic-cxr-jpg/2.1.0/mimic-cxr-2.0.0-metadata.csv.gz"),
    ("chexpert.csv", "https://physionet.org/files/mimic-cxr-jpg/2.1.0/mimic-cxr-2.0.0-chexpert.csv.gz"),
]

for target_name, url in metadata_files:
    if not os.path.exists(target_name):
        gz_name = target_name + ".gz"
        subprocess.run("wget -q --netrc -O " + gz_name + " " + url, shell=True, check=True)
        subprocess.run("gunzip -f " + gz_name, shell=True, check=True)

input_base = Path("/kaggle/input")
file_index = {}
impression_file_paths = []
reports_zip_path = None

for root, _, files in os.walk(input_base):
    for f in files:
        full_path = os.path.join(root, f)
        if f in ["mimic_train_impressions.csv", "train_imp.csv", "mimic_test_impressions.csv", "test_imp.csv"]:
            if full_path not in impression_file_paths:
                impression_file_paths.append(full_path)
        elif f == "mimic-cxr-reports.zip":
            reports_zip_path = full_path
        elif f.endswith(".jpg"):
            parts = full_path.split("/")
            rel_path = None
            for prefix in ("p10", "p11"):
                if prefix in parts:
                    idx = parts.index(prefix)
                    rel_path = "/".join(parts[idx:])
                    break
            if rel_path is not None and rel_path not in file_index:
                file_index[rel_path] = full_path

if not impression_file_paths:
    sys.exit("Error: No report impression CSV files found in /kaggle/input/.")

TOTAL_REFERENCE_POOL = 75216
indexed_count = len(file_index)
coverage_pct = (indexed_count / TOTAL_REFERENCE_POOL) * 100.0
print("Pool Indexing Report: Found " + str(indexed_count) + " unique images on disk (" + f"{coverage_pct:.1f}%" + " of 75,216 reference pool).")

report_dfs = [pd.read_csv(p) for p in impression_file_paths]
report_df = pd.concat(report_dfs, ignore_index=True).drop_duplicates(subset=["dicom_id"])

meta_df = pd.read_csv("meta.csv")
chex_df = pd.read_csv("chexpert.csv")

chex_label_cols = [c for c in chex_df.columns if c not in ["subject_id", "study_id"]]
for c in chex_label_cols:
    chex_df[c] = chex_df[c].fillna(0.0)
    chex_df[c] = chex_df[c].replace(-1.0, 0.0)

mask_cohort = (meta_df["subject_id"] >= 10000000) & (meta_df["subject_id"] < 12000000)
meta_cohort = meta_df[mask_cohort].copy()

merged_df = pd.merge(meta_cohort, chex_df[["study_id"] + chex_label_cols], on="study_id", how="inner")
merged_df = pd.merge(merged_df, report_df[["dicom_id", "report"]], on="dicom_id", how="left")
merged_df = merged_df.dropna(subset=["report"]).reset_index(drop=True)

def get_path(row):
    subject_id = int(row["subject_id"])
    study_id = int(row["study_id"])
    prefix = "p10" if subject_id < 11000000 else "p11"
    rel = prefix + "/p" + str(subject_id) + "/s" + str(study_id) + "/" + str(row["dicom_id"]) + ".jpg"
    return file_index.get(rel, None)

merged_df["image_path"] = merged_df.apply(get_path, axis=1)
merged_df = merged_df.dropna(subset=["image_path"]).reset_index(drop=True)

if reports_zip_path:
    print("Extracting and parsing reports from mimic-cxr-reports.zip...")
    study_map = {}
    for idx, row in merged_df.iterrows():
        study_map[(int(row["subject_id"]), int(row["study_id"]))] = idx
        
    reports_dict = {}
    with zipfile.ZipFile(reports_zip_path) as z:
        for name in z.namelist():
            if name.endswith(".txt"):
                parts = name.split("/")
                if len(parts) >= 4:
                    try:
                        sub_id = int(parts[2][1:])
                        std_id = int(parts[3][1:-4])
                        if (sub_id, std_id) in study_map:
                            with zipfile.ZipFile(reports_zip_path).open(name) as f:
                                text = f.read().decode('utf-8', errors='ignore')
                                lower_text = text.lower()
                                imp_idx = lower_text.find("impression:")
                                if imp_idx != -1:
                                    imp = text[imp_idx + len("impression:"):].strip()
                                else:
                                    imp = text.strip()
                                reports_dict[study_map[(sub_id, std_id)]] = " ".join(imp.split())
                    except Exception:
                        pass
                        
    replaced_count = 0
    for idx, text in reports_dict.items():
        if text:
            merged_df.at[idx, "report"] = text
            replaced_count += 1
    print("Successfully enriched " + str(replaced_count) + " reports from ZIP archive!")

np.random.seed(42)
gss_train = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, temp_idx = next(gss_train.split(merged_df, groups=merged_df["subject_id"]))

train_df = merged_df.iloc[train_idx].copy()
temp_df = merged_df.iloc[temp_idx].copy()

gss_val = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
val_idx, test_idx = next(gss_val.split(temp_df, groups=temp_df["subject_id"]))

val_df = temp_df.iloc[val_idx].copy()
test_df = temp_df.iloc[test_idx].copy()

train_df["split"] = "train"
val_df["split"] = "val"
test_df["split"] = "test"

master_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
master_df.to_csv("/kaggle/working/mimic_master_split.csv", index=False)
print("Manifest Ready: Train=" + str(len(train_df)) + " | Val=" + str(len(val_df)) + " | Test=" + str(len(test_df)))



Pool Indexing Report: Found 73995 unique images on disk (98.4% of 75,216 reference pool).


Manifest Ready: Train=50176 | Val=6190 | Test=6388


In [3]:
# -------------------------------------------------------------------------
# 2. RAD-DINO Extraction, 4D Prefix Attention, Qwen-2.5-3B QLoRA Training
# -------------------------------------------------------------------------
import os, gc, time, math, random, json, torch, faiss, numpy as np, pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
from transformers import (
    AutoModel, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
    get_cosine_schedule_with_warmup
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, get_peft_model_state_dict
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using compute device: {device}")

def log_vram(prefix=""):
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / (1024 ** 2)
        res = torch.cuda.memory_reserved() / (1024 ** 2)
        print(f"[VRAM Monitor {prefix}] Allocated: {alloc:.1f} MB | Reserved: {res:.1f} MB", flush=True)

log_vram("Initial")

manifest_path = "/kaggle/working/mimic_master_split.csv"
df = pd.read_csv(manifest_path)
train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df = df[df["split"] == "val"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)

CHEXPERT_COLS = [
    "Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Enlarged Cardiomediastinum",
    "Fracture", "Lung Lesion", "Lung Opacity", "No Finding", "Pleural Effusion",
    "Pleural Other", "Pneumonia", "Pneumothorax", "Support Devices"
]

pos_counts = train_df[CHEXPERT_COLS].sum().values
neg_counts = len(train_df) - pos_counts
pos_weights = np.clip(neg_counts / np.maximum(pos_counts, 1.0), 1.0, 10.0)
pos_weights_tensor = torch.tensor(pos_weights, dtype=torch.float, device=device)
print(f"Computed Clinical Pathology Pos-Weights: {np.round(pos_weights, 2)}", flush=True)

# -------------------------------------------------------------------------
# 1. Memory-Safe RAD-DINO Feature Extraction (Skip if already on disk)
# -------------------------------------------------------------------------
train_feat_path = "/kaggle/working/features_spatial_train.dat"
if not os.path.exists(train_feat_path):
    print("Loading Microsoft RAD-DINO extractor from Hugging Face...", flush=True)
    class RadDinoExtractor(nn.Module):
        def __init__(self):
            super().__init__()
            self.model = AutoModel.from_pretrained("microsoft/rad-dino")
            self.model.eval()
            
        @torch.no_grad()
        def forward(self, x):
            with torch.amp.autocast("cuda"):
                outputs = self.model(x)
                hidden = outputs.last_hidden_state
                cls_token = hidden[:, 0, :]
                patch_tokens = hidden[:, 1:, :]
                spatial_128 = F.adaptive_avg_pool1d(patch_tokens.transpose(1, 2), 128).transpose(1, 2)
            return spatial_128, cls_token

    raddino_extractor = RadDinoExtractor().to(device)

    rad_transform = transforms.Compose([
        transforms.Resize(518, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.CenterCrop(518),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5307, 0.5307, 0.5307], std=[0.2583, 0.2583, 0.2583])
    ])

    class FeatureExtractionDataset(Dataset):
        def __init__(self, df, transform):
            self.df = df
            self.transform = transform
        def __len__(self):
            return len(self.df)
        def __getitem__(self, idx):
            path = self.df.iloc[idx]["image_path"]
            try:
                img = Image.open(path).convert("RGB")
                tensor = self.transform(img)
            except:
                tensor = torch.zeros(3, 518, 518)
            return tensor

    def extract_features_to_memmap(df_split, split_name, batch_size=32):
        n_samples = len(df_split)
        spatial_file = f"/kaggle/working/features_spatial_{split_name}.dat"
        mean_file = f"/kaggle/working/features_mean_{split_name}.dat"
        spatial_mmap = np.memmap(spatial_file, dtype="float16", mode="w+", shape=(n_samples, 128, 768))
        mean_mmap = np.memmap(mean_file, dtype="float32", mode="w+", shape=(n_samples, 768))
        ds = FeatureExtractionDataset(df_split, rad_transform)
        loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
        idx_cursor = 0
        print(f"Extracting RAD-DINO features for split: {split_name} (N={n_samples}) in batches of {batch_size}...", flush=True)
        for batch_tensors in tqdm(loader, desc=f"RAD-DINO Features [{split_name}]"):
            b_size = batch_tensors.size(0)
            batch_tensors = batch_tensors.to(device)
            spatial_tokens, mean_feats = raddino_extractor(batch_tensors)
            spatial_mmap[idx_cursor:idx_cursor + b_size] = spatial_tokens.cpu().to(torch.float16).numpy()
            mean_mmap[idx_cursor:idx_cursor + b_size] = mean_feats.cpu().to(torch.float32).numpy()
            idx_cursor += b_size
        spatial_mmap.flush()
        mean_mmap.flush()
        print(f"Split {split_name} extraction complete: {n_samples} samples saved.", flush=True)

    extract_features_to_memmap(train_df, "train", batch_size=32)
    extract_features_to_memmap(val_df, "val", batch_size=32)
    extract_features_to_memmap(test_df, "test", batch_size=32)

    del raddino_extractor
    torch.cuda.empty_cache()
    gc.collect()
else:
    print("Precomputed RAD-DINO features found on disk. Skipping extraction phase.", flush=True)

log_vram("After Feature Extraction")

# -------------------------------------------------------------------------
# 2. Strict Patient-Leakage-Free FAISS Index & RAG Pre-calculation
# -------------------------------------------------------------------------
enriched_manifest_path = "/kaggle/working/mimic_master_split_with_rag.csv"
if not os.path.exists(enriched_manifest_path):
    print("Building FAISS FlatIP Index...", flush=True)
    train_mean_mmap = np.memmap("/kaggle/working/features_mean_train.dat", dtype="float32", mode="r", shape=(len(train_df), 768))
    train_vectors = np.array(train_mean_mmap)
    faiss.normalize_L2(train_vectors)
    dim = 768
    index = faiss.IndexFlatIP(dim)
    index.add(train_vectors)
    print(f"FAISS Index Ready with {index.ntotal} training exemplar vectors.", flush=True)

    def compute_rag_retrievals(split_df, mean_mmap_file, is_train=False):
        mean_mmap = np.memmap(mean_mmap_file, dtype="float32", mode="r", shape=(len(split_df), 768))
        query_vectors = np.array(mean_mmap)
        faiss.normalize_L2(query_vectors)
        k_search = 50 if is_train else 10
        D, I = index.search(query_vectors, k_search)
        retrieved_reports = []
        for i in range(len(split_df)):
            curr_subj = split_df.iloc[i]["subject_id"]
            chosen_idx = None
            if is_train:
                for neighbor_idx in I[i]:
                    cand_subj = train_df.iloc[neighbor_idx]["subject_id"]
                    if cand_subj != curr_subj:
                        chosen_idx = neighbor_idx
                        break
                if chosen_idx is None:
                    chosen_idx = I[i, 1]
            else:
                chosen_idx = I[i, 0]
                
            raw_rep = train_df.iloc[chosen_idx]["report"]
            if pd.isna(raw_rep) or str(raw_rep).strip().lower() in ["", "nan", "none"]:
                clean_rep = "Normal chest radiograph with no acute cardiopulmonary abnormality."
            else:
                clean_rep = str(raw_rep).strip()
            retrieved_reports.append(clean_rep)
            
        split_df = split_df.copy()
        split_df["retrieved_report"] = retrieved_reports
        return split_df

    train_df = compute_rag_retrievals(train_df, "/kaggle/working/features_mean_train.dat", is_train=True)
    val_df = compute_rag_retrievals(val_df, "/kaggle/working/features_mean_val.dat", is_train=False)
    test_df = compute_rag_retrievals(test_df, "/kaggle/working/features_mean_test.dat", is_train=False)

    enriched_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
    enriched_df.to_csv(enriched_manifest_path, index=False)
    print(f"Saved RAG-enriched manifest to {enriched_manifest_path}", flush=True)
else:
    print("Found precomputed RAG manifest. Loading from disk...", flush=True)
    enriched_df = pd.read_csv(enriched_manifest_path)
    train_df = enriched_df[enriched_df["split"] == "train"].reset_index(drop=True)
    val_df = enriched_df[enriched_df["split"] == "val"].reset_index(drop=True)
    test_df = enriched_df[enriched_df["split"] == "test"].reset_index(drop=True)

# -------------------------------------------------------------------------
# 3. Load Qwen-2.5-3B-Instruct with 4-bit QLoRA
# -------------------------------------------------------------------------
print("Loading Qwen-2.5-3B-Instruct with 4-bit Quantization (BitsAndBytes NF4)...", flush=True)
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

base_qwen = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map={"": 0},
    trust_remote_code=True
)

base_qwen.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
base_qwen.config.use_cache = False
base_qwen = prepare_model_for_kbit_training(base_qwen)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

qwen_lora_model = get_peft_model(base_qwen, peft_config)
qwen_lora_model.print_trainable_parameters()
log_vram("After Qwen-3B QLoRA Load")

# -------------------------------------------------------------------------
# 4. Multi-Task Vision-Language Model with 4D Prefix Attention
# -------------------------------------------------------------------------
class MultiTaskQwenModel(nn.Module):
    def __init__(self, qwen_model, visual_dim=768, qwen_dim=2048, num_patches=128):
        super().__init__()
        self.qwen = qwen_model
        self.num_patches = num_patches
        self.qwen_dim = qwen_dim
        
        self.visual_projection = nn.Sequential(
            nn.Linear(visual_dim, qwen_dim),
            nn.GELU(),
            nn.Linear(qwen_dim, qwen_dim)
        ).to(device=device)
        
        self.classifier = nn.Sequential(
            nn.Linear(visual_dim, 512),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(512, 14)
        ).to(device=device)
        
    def build_4d_prefix_attention_mask(self, batch_size, seq_len, device, dtype, text_pad_mask=None):
        total_len = self.num_patches + seq_len
        mask = torch.full((total_len, total_len), -10000.0, dtype=dtype, device=device)
        mask[:self.num_patches, :self.num_patches] = 0.0
        for i in range(self.num_patches, total_len):
            mask[i, :i+1] = 0.0
            
        mask_4d = mask.unsqueeze(0).unsqueeze(0).expand(batch_size, 1, total_len, total_len).clone()
        if text_pad_mask is not None:
            pad_expanded = text_pad_mask.unsqueeze(1).unsqueeze(2)
            mask_4d[:, :, :, self.num_patches:] = mask_4d[:, :, :, self.num_patches:].masked_fill(pad_expanded == 0, -10000.0)
        return mask_4d
        
    def forward(self, spatial_tokens, mean_features, input_ids, labels=None, text_pad_mask=None):
        batch_size = spatial_tokens.size(0)
        seq_len = input_ids.size(1)
        
        cls_logits = self.classifier(mean_features.float())
        visual_embeds = self.visual_projection(spatial_tokens.float())
        text_embeds = self.qwen.get_input_embeddings()(input_ids)
        
        if visual_embeds.dtype != text_embeds.dtype:
            visual_embeds = visual_embeds.to(text_embeds.dtype)
            
        inputs_embeds = torch.cat([visual_embeds, text_embeds], dim=1)
        attention_mask_4d = self.build_4d_prefix_attention_mask(batch_size, seq_len, spatial_tokens.device, inputs_embeds.dtype, text_pad_mask)
        
        if labels is not None:
            vis_labels = torch.full((batch_size, self.num_patches), -100, dtype=labels.dtype, device=labels.device)
            combined_labels = torch.cat([vis_labels, labels], dim=1)
        else:
            combined_labels = None
            
        outputs = self.qwen(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask_4d,
            labels=combined_labels
        )
        return outputs.loss, cls_logits
        
    @torch.no_grad()
    def generate_report(self, spatial_token, retrieved_report, max_new_tokens=60, repetition_penalty=1.15, max_rag_tokens=40):
        self.eval()
        self.qwen.config.use_cache = True
        
        if spatial_token.dim() == 2:
            spatial_token = spatial_token.unsqueeze(0)
            
        visual_embeds = self.visual_projection(spatial_token.float())
        
        if pd.isna(retrieved_report) or str(retrieved_report).strip().lower() in ["", "nan", "none"]:
            retrieved_clean = "Normal chest radiograph with no acute cardiopulmonary abnormality."
        else:
            retrieved_clean = str(retrieved_report).strip()
            
        # Truncate RAG context to ensure prompt stays compact
        rag_tokens = tokenizer.encode(retrieved_clean, add_special_tokens=False)[:max_rag_tokens]
        retrieved_clean = tokenizer.decode(rag_tokens, skip_special_tokens=True)
            
        messages = [
            {"role": "system", "content": "You are a professional radiologist assistant. Generate a concise, diagnostic chest X-ray findings impression report based on the image features and reference exemplar."},
            {"role": "user", "content": f"Reference Report:\n{retrieved_clean}\n\nGenerate the chest X-ray findings report:"}
        ]
        prompt_str = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        prompt_enc = tokenizer(prompt_str, return_tensors="pt").to(device)
        prompt_ids = prompt_enc["input_ids"]
        prompt_embeds = self.qwen.get_input_embeddings()(prompt_ids)
        
        if visual_embeds.dtype != prompt_embeds.dtype:
            visual_embeds = visual_embeds.to(prompt_embeds.dtype)
            
        inputs_embeds = torch.cat([visual_embeds, prompt_embeds], dim=1)
        seq_len = prompt_ids.size(1)
        
        attention_mask_4d = self.build_4d_prefix_attention_mask(1, seq_len, spatial_token.device, inputs_embeds.dtype, prompt_enc["attention_mask"])
        
        prefill_out = self.qwen(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask_4d,
            use_cache=True
        )
        past_key_values = prefill_out.past_key_values
        next_token_logits = prefill_out.logits[:, -1, :].clone()
        
        for token_id in set(prompt_ids[0].tolist()):
            if next_token_logits[0, token_id] > 0:
                next_token_logits[0, token_id] /= repetition_penalty
            else:
                next_token_logits[0, token_id] *= repetition_penalty
                
        curr_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)
        generated_token_ids = [curr_token.item()]
        
        for _ in range(max_new_tokens - 1):
            if curr_token.item() in [tokenizer.eos_token_id, tokenizer.pad_token_id, 151645]:
                break
            step_out = self.qwen(
                input_ids=curr_token,
                past_key_values=past_key_values,
                use_cache=True
            )
            past_key_values = step_out.past_key_values
            next_logits = step_out.logits[:, -1, :].clone()
            
            for token_id in set(generated_token_ids):
                if next_logits[0, token_id] > 0:
                    next_logits[0, token_id] /= repetition_penalty
                else:
                    next_logits[0, token_id] *= repetition_penalty
                    
            curr_token = torch.argmax(next_logits, dim=-1, keepdim=True)
            generated_token_ids.append(curr_token.item())
            
        self.qwen.config.use_cache = False
        pred_text = tokenizer.decode(generated_token_ids, skip_special_tokens=True).strip()
        return pred_text

model = MultiTaskQwenModel(qwen_lora_model).to(device)
log_vram("After Multimodal Model Assembly")

# -------------------------------------------------------------------------
# 5. Robust Sample-Preserving Dataset (max_seq_len=160, MAX_RAG_TOKENS=40)
# -------------------------------------------------------------------------
class MultiTaskQwenDataset(Dataset):
    def __init__(self, df_split, split_name, tokenizer, max_seq_len=160, max_samples=None, max_rag_tokens=40):
        df_split_with_idx = df_split.copy().reset_index(drop=True)
        df_split_with_idx["orig_split_idx"] = np.arange(len(df_split_with_idx))
        
        if max_samples is not None and len(df_split_with_idx) > max_samples:
            # Truly random, diverse sampling across all patients
            self.df = df_split_with_idx.sample(n=max_samples, random_state=42).reset_index(drop=True)
        else:
            self.df = df_split_with_idx
            
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len
        self.max_rag_tokens = max_rag_tokens
        n_total_split = len(df_split)
        self.spatial_mmap = np.memmap(f"/kaggle/working/features_spatial_{split_name}.dat", dtype="float16", mode="r", shape=(n_total_split, 128, 768))
        self.mean_mmap = np.memmap(f"/kaggle/working/features_mean_{split_name}.dat", dtype="float32", mode="r", shape=(n_total_split, 768))
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        orig_idx = int(row["orig_split_idx"])
        spatial_token = torch.from_numpy(np.array(self.spatial_mmap[orig_idx]))
        mean_feat = torch.from_numpy(np.array(self.mean_mmap[orig_idx]))
        cls_labels = torch.tensor([row[c] for c in CHEXPERT_COLS], dtype=torch.float)
        
        # 1. RAG Context with explicit 40-token truncation
        retrieved_raw = row["retrieved_report"]
        if pd.isna(retrieved_raw) or str(retrieved_raw).strip().lower() in ["", "nan", "none"]:
            retrieved_clean = "Normal chest radiograph with no acute cardiopulmonary abnormality."
        else:
            retrieved_clean = str(retrieved_raw).strip()
            
        rag_tokens = self.tokenizer.encode(retrieved_clean, add_special_tokens=False)[:self.max_rag_tokens]
        retrieved_clean = self.tokenizer.decode(rag_tokens, skip_special_tokens=True)
            
        # 2. Strict Target Report Guard
        raw_target = row["report"]
        if pd.isna(raw_target) or str(raw_target).strip().lower() in ["", "nan", "none"]:
            target_report = "Normal chest radiograph with no acute cardiopulmonary abnormality."
        else:
            target_report = str(raw_target).strip()
        
        messages = [
            {"role": "system", "content": "You are a professional radiologist assistant. Generate a concise, diagnostic chest X-ray findings impression report based on the image features and reference exemplar."},
            {"role": "user", "content": f"Reference Report:\n{retrieved_clean}\n\nGenerate the chest X-ray findings report:"}
        ]
        prompt_str = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        full_text = prompt_str + target_report + "<|im_end|>"
        
        prompt_tokens = self.tokenizer.encode(prompt_str, add_special_tokens=False)
        full_enc = self.tokenizer(full_text, max_length=self.max_seq_len, padding="max_length", truncation=True, return_tensors="pt")
        
        input_ids = full_enc["input_ids"].squeeze(0)
        attention_mask = full_enc["attention_mask"].squeeze(0)
        
        labels = input_ids.clone()
        prompt_len = min(len(prompt_tokens), self.max_seq_len)
        labels[:prompt_len] = -100
        labels[labels == self.tokenizer.pad_token_id] = -100
        
        return {
            "spatial_token": spatial_token,
            "mean_feat": mean_feat,
            "cls_labels": cls_labels,
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }

# Optimal budget: 15,000 diverse training samples + 2,000 validation samples
TRAIN_SAMPLE_BUDGET = 15000
train_ds = MultiTaskQwenDataset(train_df, "train", tokenizer, max_seq_len=160, max_samples=TRAIN_SAMPLE_BUDGET, max_rag_tokens=40)
val_ds = MultiTaskQwenDataset(val_df, "val", tokenizer, max_seq_len=160, max_samples=2000, max_rag_tokens=40)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=0, pin_memory=False)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=0, pin_memory=False)

# -------------------------------------------------------------------------
# 6. Training Loop (2 Epochs = 3,750 Optimizer Steps)
# -------------------------------------------------------------------------
epochs = 2
accumulation_steps = 4

trainable_params = [
    {"params": model.visual_projection.parameters(), "lr": 1e-4, "weight_decay": 0.01},
    {"params": model.classifier.parameters(), "lr": 1e-4, "weight_decay": 0.01},
    {"params": [p for p in model.qwen.parameters() if p.requires_grad], "lr": 5e-5, "weight_decay": 0.01}
]
optimizer = torch.optim.AdamW(trainable_params)

total_steps = (len(train_loader) // accumulation_steps) * epochs
warmup_steps = int(0.10 * total_steps)
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

criterion_cls = nn.BCEWithLogitsLoss(pos_weight=pos_weights_tensor)
best_val_loss = float("inf")
checkpoint_path = "/kaggle/working/model4_qwen_best.pt"

training_history = {
    "epoch": [],
    "train_joint_loss": [], "train_text_loss": [], "train_cls_loss": [],
    "val_joint_loss": [], "val_text_loss": [], "val_cls_loss": []
}

print(f"\n{'='*70}", flush=True)
print(f"Starting Multi-Task Qwen-3B Training ({epochs} Epochs, {len(train_loader)} Batches/epoch, {total_steps} Optimizer Steps)", flush=True)
print(f"{'='*70}\n", flush=True)

start_train_time = time.time()

for epoch in range(1, epochs + 1):
    model.train()
    running_joint_loss = 0.0
    running_text_loss = 0.0
    running_cls_loss = 0.0
    
    optimizer.zero_grad()
    epoch_start_time = time.time()
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs} [Train]")
    
    for step, batch in enumerate(train_pbar):
        spatial_t = batch["spatial_token"].to(device)
        mean_f = batch["mean_feat"].to(device)
        cls_labels = batch["cls_labels"].to(device)
        input_ids = batch["input_ids"].to(device)
        attn_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        loss_text, cls_logits = model(spatial_t, mean_f, input_ids, labels=labels, text_pad_mask=attn_mask)
        loss_cls = criterion_cls(cls_logits, cls_labels)
        joint_loss = loss_text + 1.0 * loss_cls
        loss_scaled = joint_loss / accumulation_steps
        
        loss_scaled.backward()
        
        if (step + 1) % accumulation_steps == 0 or (step + 1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            
        running_joint_loss += joint_loss.item()
        running_text_loss += loss_text.item()
        running_cls_loss += loss_cls.item()
        
        if (step + 1) % 50 == 0 or (step + 1) == len(train_loader):
            curr_lr = scheduler.get_last_lr()[0]
            step_pct = ((step + 1) / len(train_loader)) * 100.0
            print(f"[Epoch {epoch}/{epochs} | Step {step+1}/{len(train_loader)} ({step_pct:.1f}%)] Joint: {running_joint_loss/(step+1):.4f} | Text: {running_text_loss/(step+1):.4f} | Cls: {running_cls_loss/(step+1):.4f} | LR: {curr_lr:.2e}", flush=True)
            
    epoch_train_time = time.time() - epoch_start_time
    avg_train_loss = running_joint_loss / len(train_loader)
    avg_train_text = running_text_loss / len(train_loader)
    avg_train_cls = running_cls_loss / len(train_loader)
    
    # Validation Loop
    model.eval()
    val_joint_loss = 0.0
    val_text_loss = 0.0
    val_cls_loss = 0.0
    
    print(f"\nEvaluating on Validation Set ({len(val_loader)} batches)...", flush=True)
    with torch.no_grad():
        for val_step, batch in enumerate(tqdm(val_loader, desc=f"Epoch {epoch}/{epochs} [Val]")):
            spatial_t = batch["spatial_token"].to(device)
            mean_f = batch["mean_feat"].to(device)
            cls_labels = batch["cls_labels"].to(device)
            input_ids = batch["input_ids"].to(device)
            attn_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            
            loss_text, cls_logits = model(spatial_t, mean_f, input_ids, labels=labels, text_pad_mask=attn_mask)
            loss_cls = criterion_cls(cls_logits, cls_labels)
            joint_loss = loss_text + 1.0 * loss_cls
            
            val_joint_loss += joint_loss.item()
            val_text_loss += loss_text.item()
            val_cls_loss += loss_cls.item()
            
    avg_val_loss = val_joint_loss / len(val_loader)
    avg_val_text = val_text_loss / len(val_loader)
    avg_val_cls = val_cls_loss / len(val_loader)
    
    training_history["epoch"].append(epoch)
    training_history["train_joint_loss"].append(avg_train_loss)
    training_history["train_text_loss"].append(avg_train_text)
    training_history["train_cls_loss"].append(avg_train_cls)
    training_history["val_joint_loss"].append(avg_val_loss)
    training_history["val_text_loss"].append(avg_val_text)
    training_history["val_cls_loss"].append(avg_val_cls)
    
    print(f"\n--- Epoch {epoch}/{epochs} Summary (Time: {epoch_train_time/60:.1f} min) ---", flush=True)
    print(f"Train Loss : Joint = {avg_train_loss:.4f} | Text = {avg_train_text:.4f} | Cls = {avg_train_cls:.4f}", flush=True)
    print(f"Val Loss   : Joint = {avg_val_loss:.4f} | Text = {avg_val_text:.4f} | Cls = {avg_val_cls:.4f}", flush=True)
    log_vram(f"Epoch {epoch} End")
    
    print("\n[Qualitative Validation Sample Check]:", flush=True)
    for sample_idx in [0, 1]:
        sample_row = val_df.iloc[sample_idx]
        sample_spat = torch.from_numpy(np.array(val_ds.spatial_mmap[sample_idx])).to(device)
        sample_pred = model.generate_report(sample_spat, sample_row["retrieved_report"], max_new_tokens=60, max_rag_tokens=40)
        print(f"  Sample {sample_idx+1} Ground Truth : {sample_row['report'][:100]}...", flush=True)
        print(f"  Sample {sample_idx+1} Generated    : {sample_pred[:100]}...", flush=True)
        print("  " + "-"*50, flush=True)
        
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        lora_state = get_peft_model_state_dict(qwen_lora_model)
        torch.save({
            "visual_projection": model.visual_projection.state_dict(),
            "classifier": model.classifier.state_dict(),
            "lora_state": lora_state,
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "epoch": epoch,
            "val_loss": best_val_loss
        }, checkpoint_path)
        print(">>> Best Checkpoint Saved Successfully <<<", flush=True)
    print("=" * 70, flush=True)

total_train_elapsed = time.time() - start_train_time
print(f"Training Complete in {total_train_elapsed/3600:.2f} hours. Best Checkpoint saved to: {checkpoint_path}", flush=True)

# -------------------------------------------------------------------------
# Save Training History JSON & Plot Learning Curves
# -------------------------------------------------------------------------
with open("/kaggle/working/training_history.json", "w", encoding="utf-8") as f:
    json.dump(training_history, f, indent=2)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
ep_range = training_history["epoch"]

axes[0].plot(ep_range, training_history["train_joint_loss"], "o-", label="Train Joint Loss", color="#1f77b4")
axes[0].plot(ep_range, training_history["val_joint_loss"], "s--", label="Val Joint Loss", color="#ff7f0e")
axes[0].set_title("Joint Multi-Task Loss", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(True, linestyle="--", alpha=0.6)
axes[0].legend()

axes[1].plot(ep_range, training_history["train_text_loss"], "o-", label="Train Text Loss", color="#2ca02c")
axes[1].plot(ep_range, training_history["val_text_loss"], "s--", label="Val Text Loss", color="#d62728")
axes[1].set_title("Causal Text Generation Loss", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Cross Entropy Loss")
axes[1].grid(True, linestyle="--", alpha=0.6)
axes[1].legend()

axes[2].plot(ep_range, training_history["train_cls_loss"], "o-", label="Train Cls Loss", color="#9467bd")
axes[2].plot(ep_range, training_history["val_cls_loss"], "s--", label="Val Cls Loss", color="#8c564b")
axes[2].set_title("Pathology Classification Loss", fontsize=13, fontweight="bold")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Weighted BCE Loss")
axes[2].grid(True, linestyle="--", alpha=0.6)
axes[2].legend()

plt.tight_layout()
plt.savefig("/kaggle/working/learning_curves.png", dpi=300)
plt.close()
print("Learning Curves successfully saved to /kaggle/working/learning_curves.png", flush=True)



Using compute device: cuda
[VRAM Monitor Initial] Allocated: 0.0 MB | Reserved: 0.0 MB


Computed Clinical Pathology Pos-Weights: [ 5.65  5.63 10.    9.34 10.   10.   10.    4.63  1.41  4.38 10.   10.
 10.    4.44]


Loading Microsoft RAD-DINO extractor from Hugging Face...


config.json:   0%|          | 0.00/879 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

... (132 outputs left out)

  --------------------------------------------------


>>> Best Checkpoint Saved Successfully <<<


Training Complete in 5.83 hours. Best Checkpoint saved to: /kaggle/working/model4_qwen_best.pt


Learning Curves successfully saved to /kaggle/working/learning_curves.png


In [4]:
# -------------------------------------------------------------------------
# 3. Final Standalone Benchmark Evaluation with Confusion Heatmaps & Artifact Exports
# -------------------------------------------------------------------------
import os, sys, math, random, time, re, json, torch, numpy as np, pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, set_peft_model_state_dict
from nltk.translate.bleu_score import sentence_bleu, corpus_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Evaluating Model 4 (RAD-DINO + Qwen-2.5-3B-Instruct) on: {device}", flush=True)

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

CHEXPERT_COLS = [
    "Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Enlarged Cardiomediastinum",
    "Fracture", "Lung Lesion", "Lung Opacity", "No Finding", "Pleural Effusion",
    "Pleural Other", "Pneumonia", "Pneumothorax", "Support Devices"
]

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

base_qwen = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map={"": 0},
    trust_remote_code=True
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
qwen_lora_model = get_peft_model(base_qwen, peft_config)

class MultiTaskQwenModel(nn.Module):
    def __init__(self, qwen_model, visual_dim=768, qwen_dim=2048, num_patches=128):
        super().__init__()
        self.qwen = qwen_model
        self.num_patches = num_patches
        self.qwen_dim = qwen_dim
        
        self.visual_projection = nn.Sequential(
            nn.Linear(visual_dim, qwen_dim),
            nn.GELU(),
            nn.Linear(qwen_dim, qwen_dim)
        ).to(device=device)
        
        self.classifier = nn.Sequential(
            nn.Linear(visual_dim, 512),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(512, 14)
        ).to(device=device)
        
    def build_4d_prefix_attention_mask(self, batch_size, seq_len, device, dtype, text_pad_mask=None):
        total_len = self.num_patches + seq_len
        mask = torch.full((total_len, total_len), -10000.0, dtype=dtype, device=device)
        mask[:self.num_patches, :self.num_patches] = 0.0
        for i in range(self.num_patches, total_len):
            mask[i, :i+1] = 0.0
            
        mask_4d = mask.unsqueeze(0).unsqueeze(0).expand(batch_size, 1, total_len, total_len).clone()
        if text_pad_mask is not None:
            pad_expanded = text_pad_mask.unsqueeze(1).unsqueeze(2)
            mask_4d[:, :, :, self.num_patches:] = mask_4d[:, :, :, self.num_patches:].masked_fill(pad_expanded == 0, -10000.0)
        return mask_4d
        
    @torch.no_grad()
    def generate_report(self, spatial_token, retrieved_report, max_new_tokens=60, repetition_penalty=1.15, max_rag_tokens=40):
        self.eval()
        self.qwen.config.use_cache = True
        
        if spatial_token.dim() == 2:
            spatial_token = spatial_token.unsqueeze(0)
            
        visual_embeds = self.visual_projection(spatial_token.float())
        
        if pd.isna(retrieved_report) or str(retrieved_report).strip().lower() in ["", "nan", "none"]:
            retrieved_clean = "Normal chest radiograph with no acute cardiopulmonary abnormality."
        else:
            retrieved_clean = str(retrieved_report).strip()
            
        rag_tokens = tokenizer.encode(retrieved_clean, add_special_tokens=False)[:max_rag_tokens]
        retrieved_clean = tokenizer.decode(rag_tokens, skip_special_tokens=True)
            
        messages = [
            {"role": "system", "content": "You are a professional radiologist assistant. Generate a concise, diagnostic chest X-ray findings impression report based on the image features and reference exemplar."},
            {"role": "user", "content": f"Reference Report:\n{retrieved_clean}\n\nGenerate the chest X-ray findings report:"}
        ]
        prompt_str = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        prompt_enc = tokenizer(prompt_str, return_tensors="pt").to(device)
        prompt_ids = prompt_enc["input_ids"]
        prompt_embeds = self.qwen.get_input_embeddings()(prompt_ids)
        
        if visual_embeds.dtype != prompt_embeds.dtype:
            visual_embeds = visual_embeds.to(prompt_embeds.dtype)
            
        inputs_embeds = torch.cat([visual_embeds, prompt_embeds], dim=1)
        seq_len = prompt_ids.size(1)
        
        attention_mask_4d = self.build_4d_prefix_attention_mask(1, seq_len, spatial_token.device, inputs_embeds.dtype, prompt_enc["attention_mask"])
        
        prefill_out = self.qwen(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask_4d,
            use_cache=True
        )
        past_key_values = prefill_out.past_key_values
        next_token_logits = prefill_out.logits[:, -1, :].clone()
        
        for token_id in set(prompt_ids[0].tolist()):
            if next_token_logits[0, token_id] > 0:
                next_token_logits[0, token_id] /= repetition_penalty
            else:
                next_token_logits[0, token_id] *= repetition_penalty
                
        curr_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)
        generated_token_ids = [curr_token.item()]
        
        for _ in range(max_new_tokens - 1):
            if curr_token.item() in [tokenizer.eos_token_id, tokenizer.pad_token_id, 151645]:
                break
            step_out = self.qwen(
                input_ids=curr_token,
                past_key_values=past_key_values,
                use_cache=True
            )
            past_key_values = step_out.past_key_values
            next_logits = step_out.logits[:, -1, :].clone()
            
            for token_id in set(generated_token_ids):
                if next_logits[0, token_id] > 0:
                    next_logits[0, token_id] /= repetition_penalty
                else:
                    next_logits[0, token_id] *= repetition_penalty
                    
            curr_token = torch.argmax(next_logits, dim=-1, keepdim=True)
            generated_token_ids.append(curr_token.item())
            
        self.qwen.config.use_cache = False
        pred_text = tokenizer.decode(generated_token_ids, skip_special_tokens=True).strip()
        return pred_text

model = MultiTaskQwenModel(qwen_lora_model).to(device)

checkpoint_path = "/kaggle/working/model4_qwen_best.pt"
ckpt = torch.load(checkpoint_path, map_location=device)
model.visual_projection.load_state_dict(ckpt["visual_projection"])
model.classifier.load_state_dict(ckpt["classifier"])

set_peft_model_state_dict(qwen_lora_model, ckpt["lora_state"])
model.eval()
print(f"Model 4 Checkpoint (Epoch {ckpt.get('epoch', 2)}) loaded successfully with verified PEFT LoRA state.", flush=True)

manifest_path = "/kaggle/working/mimic_master_split_with_rag.csv"
enriched_df = pd.read_csv(manifest_path)
test_df = enriched_df[enriched_df["split"] == "test"].reset_index(drop=True)
n_test = len(test_df)

test_spatial_mmap = np.memmap("/kaggle/working/features_spatial_test.dat", dtype="float16", mode="r", shape=(n_test, 128, 768))

print(f"Generating clinical reports on {n_test} test cases using Qwen-3B with True 4D Prefix Attention...", flush=True)
results = []
smooth = SmoothingFunction().method1
r_scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

references = []
hypotheses = []

eval_start_time = time.time()
for i in tqdm(range(n_test), desc="Evaluating Model 4 Test Split"):
    row = test_df.iloc[i]
    spatial_t = torch.from_numpy(np.array(test_spatial_mmap[i])).to(device)
    retrieved_r = str(row["retrieved_report"]).strip()
    
    pred_text = model.generate_report(spatial_t, retrieved_r, max_new_tokens=60, repetition_penalty=1.15, max_rag_tokens=40)
    
    raw_gt = row['report']
    if pd.isna(raw_gt) or str(raw_gt).strip().lower() in ["", "nan", "none"]:
        gt_text = "Normal chest radiograph with no acute cardiopulmonary abnormality."
    else:
        gt_text = str(raw_gt).strip()
    
    gt_toks = gt_text.lower().split()
    pr_toks = pred_text.lower().split()
    
    references.append([gt_toks])
    hypotheses.append(pr_toks)
    
    b1 = sentence_bleu([gt_toks], pr_toks, weights=(1, 0, 0, 0), smoothing_function=smooth) if len(pr_toks) > 0 else 0.0
    b2 = sentence_bleu([gt_toks], pr_toks, weights=(0.5, 0.5, 0, 0), smoothing_function=smooth) if len(pr_toks) > 0 else 0.0
    b3 = sentence_bleu([gt_toks], pr_toks, weights=(0.33, 0.33, 0.33, 0), smoothing_function=smooth) if len(pr_toks) > 0 else 0.0
    b4 = sentence_bleu([gt_toks], pr_toks, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smooth) if len(pr_toks) > 0 else 0.0
    
    rouge_res = r_scorer.score(gt_text, pred_text)
    r1 = rouge_res['rouge1'].fmeasure
    r2 = rouge_res['rouge2'].fmeasure
    rl = rouge_res['rougeL'].fmeasure
    
    res_dict = {
        "dicom_id": row["dicom_id"],
        "study_id": row["study_id"],
        "ground_truth": gt_text,
        "prediction": pred_text,
        "retrieved_prompt": retrieved_r,
        "bleu1": b1, "bleu2": b2, "bleu3": b3, "bleu4": b4,
        "rouge1": r1, "rouge2": r2, "rougeL": rl
    }
    for col in CHEXPERT_COLS:
        res_dict[f"gt_{col}"] = row[col]
    results.append(res_dict)

res_df = pd.DataFrame(results)
res_df.to_csv("/kaggle/working/model4_qwen_test_predictions.csv", index=False)

corpus_b1 = corpus_bleu(references, hypotheses, weights=(1, 0, 0, 0), smoothing_function=smooth)
corpus_b2 = corpus_bleu(references, hypotheses, weights=(0.5, 0.5, 0, 0), smoothing_function=smooth)
corpus_b3 = corpus_bleu(references, hypotheses, weights=(0.33, 0.33, 0.33, 0), smoothing_function=smooth)
corpus_b4 = corpus_bleu(references, hypotheses, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smooth)

# -------------------------------------------------------------------------
# NegEx Medical Scope Parser
# -------------------------------------------------------------------------
print("Extracting Clinical Pathology Labels using NegEx Medical Scope Parser...", flush=True)

pathology_keywords = {
    "Atelectasis": ["atelectasis", "collapse", "atelectatic", "volume loss"],
    "Cardiomegaly": ["cardiomegaly", "enlarged heart", "heart is enlarged", "cardiac enlargement", "cardiac silhouette is enlarged"],
    "Consolidation": ["consolidation", "airspace disease", "air space disease", "infiltrate", "air bronchogram"],
    "Edema": ["edema", "pulmonary edema", "fluid overload", "vascular congestion", "chf", "heart failure", "vascular fullness"],
    "Enlarged Cardiomediastinum": ["enlarged cardiomediastinum", "widened mediastinum", "mediastinal widening"],
    "Fracture": ["fracture", "fractured", "broken rib", "callus formation", "deformity"],
    "Lung Lesion": ["nodule", "nodules", "mass", "masses", "lung lesion", "neoplasm", "granuloma"],
    "Lung Opacity": ["opacity", "opacities", "hazy", "haziness", "density", "densities", "infiltrate"],
    "No Finding": ["normal", "clear", "unremarkable", "within normal limits", "no acute cardiopulmonary", "no acute pulmonary", "no active disease"],
    "Pleural Effusion": ["pleural effusion", "effusion", "pleural fluid", "blunting of costophrenic", "fluid in pleural"],
    "Pleural Other": ["pleural thickening", "fibrothorax", "pleural plaque", "pleural calcification", "apical capping"],
    "Pneumonia": ["pneumonia", "infectious process", "bronchopneumonia"],
    "Pneumothorax": ["pneumothorax", "collapsed lung", "apical pneumothorax"],
    "Support Devices": ["tube", "catheter", "pacer", "pacemaker", "hardware", "drain", "wire", "sternotomy", "picc", "et tube"]
}

PRE_NEGATIONS = [
    r"\bno\b", r"\bwithout\b", r"\bnegative for\b", r"\bdenies\b",
    r"\bfree of\b", r"\bclear of\b", r"\bno evidence of\b", r"\bno sign of\b",
    r"\bno signs of\b", r"\brule out\b", r"\br/o\b", r"\babsent\b",
    r"\bunremarkable for\b", r"\bdisclosed no\b", r"\bno convincing\b"
]

POST_NEGATIONS = [
    r"\bunremarkable\b", r"\bwithin normal limits\b", r"\bruled out\b",
    r"\bnot seen\b", r"\bwas excluded\b", r"\bis excluded\b", r"\bresolved\b"
]

def extract_clinical_labels_negex(text_list):
    labels_dict = {col: [] for col in CHEXPERT_COLS}
    for text in text_list:
        text_lower = str(text).lower()
        sentences = re.split(r'[\.\;\n\!]', text_lower)
        sample_labels = {col: 0 for col in CHEXPERT_COLS}
        
        for sent in sentences:
            sent = sent.strip()
            if not sent:
                continue
                
            for col, kw_list in pathology_keywords.items():
                if col == "No Finding":
                    if any(kw in sent for kw in kw_list):
                        sample_labels[col] = 1
                else:
                    for kw in kw_list:
                        if kw in sent:
                            is_negated = False
                            for pre in PRE_NEGATIONS:
                                pattern = pre + r".{0,40}?\b" + re.escape(kw) + r"\b"
                                if re.search(pattern, sent):
                                    is_negated = True
                                    break
                            if not is_negated:
                                for post in POST_NEGATIONS:
                                    pattern = r"\b" + re.escape(kw) + r"\b.{0,30}?" + post
                                    if re.search(pattern, sent):
                                        is_negated = True
                                        break
                            if not is_negated:
                                sample_labels[col] = 1
                                break
                                
        for col in CHEXPERT_COLS:
            labels_dict[col].append(sample_labels[col])
            
    return pd.DataFrame(labels_dict)

pred_labels_df = extract_clinical_labels_negex(res_df["prediction"].tolist())

y_true = res_df[[f"gt_{c}" for c in CHEXPERT_COLS]].values
y_pred = pred_labels_df[CHEXPERT_COLS].values

macro_f1_all14 = f1_score(y_true, y_pred, average="macro", zero_division=0)
core10_indices = [CHEXPERT_COLS.index(c) for c in CHEXPERT_COLS if c not in ["Enlarged Cardiomediastinum", "Fracture", "Pleural Other"]]
macro_f1_core10 = f1_score(y_true[:, core10_indices], y_pred[:, core10_indices], average="macro", zero_division=0)
micro_f1 = f1_score(y_true, y_pred, average="micro", zero_division=0)
micro_prec = precision_score(y_true, y_pred, average="micro", zero_division=0)
micro_rec = recall_score(y_true, y_pred, average="micro", zero_division=0)

all_bigrams = []
for p in res_df["prediction"]:
    words = str(p).lower().split()
    if len(words) >= 2:
        all_bigrams.extend([f"{words[j]} {words[j+1]}" for j in range(len(words)-1)])
distinct_2 = len(set(all_bigrams)) / max(len(all_bigrams), 1)

# -------------------------------------------------------------------------
# Save Structured Benchmark Reports (JSON & CSV)
# -------------------------------------------------------------------------
benchmark_metrics = {
    "model_name": "Model 4 (RAD-DINO + Qwen-2.5-3B + QLoRA + RAG)",
    "sentence_bleu1": float(res_df['bleu1'].mean()),
    "sentence_bleu2": float(res_df['bleu2'].mean()),
    "sentence_bleu3": float(res_df['bleu3'].mean()),
    "sentence_bleu4": float(res_df['bleu4'].mean()),
    "corpus_bleu1": float(corpus_b1),
    "corpus_bleu2": float(corpus_b2),
    "corpus_bleu3": float(corpus_b3),
    "corpus_bleu4": float(corpus_b4),
    "rouge1": float(res_df['rouge1'].mean()),
    "rouge2": float(res_df['rouge2'].mean()),
    "rougeL": float(res_df['rougeL'].mean()),
    "distinct_2": float(distinct_2),
    "clinical_macro_f1_all14": float(macro_f1_all14),
    "clinical_macro_f1_core10": float(macro_f1_core10),
    "clinical_micro_f1": float(micro_f1),
    "clinical_micro_precision": float(micro_prec),
    "clinical_micro_recall": float(micro_rec)
}

with open("/kaggle/working/test_metrics.json", "w", encoding="utf-8") as f:
    json.dump(benchmark_metrics, f, indent=2)

pathology_rows = []
for idx, col in enumerate(CHEXPERT_COLS):
    yt = y_true[:, idx]
    yp = y_pred[:, idx]
    f1 = f1_score(yt, yp, zero_division=0)
    prec = precision_score(yt, yp, zero_division=0)
    rec = recall_score(yt, yp, zero_division=0)
    tp = int(((yp == 1) & (yt == 1)).sum())
    fp = int(((yp == 1) & (yt == 0)).sum())
    fn = int(((yp == 0) & (yt == 1)).sum())
    tn = int(((yp == 0) & (yt == 0)).sum())
    flag = "Low Sample (N < 150)" if yt.sum() < 150 else "OK"
    pathology_rows.append({
        "pathology": col,
        "true_positives": tp,
        "false_positives": fp,
        "false_negatives": fn,
        "true_negatives": tn,
        "precision": prec,
        "recall": rec,
        "f1_score": f1,
        "sample_flag": flag
    })

pathology_breakdown_df = pd.DataFrame(pathology_rows)
pathology_breakdown_df.to_csv("/kaggle/working/pathology_breakdown.csv", index=False)

print("\n" + "="*80, flush=True)
print("OVERALL BENCHMARK: Ultimate Model 4 (RAD-DINO + Qwen-2.5-3B-Instruct + QLoRA)", flush=True)
print("="*80, flush=True)
print(f"Sentence BLEU-1                | {benchmark_metrics['sentence_bleu1']:.4f}", flush=True)
print(f"Sentence BLEU-2                | {benchmark_metrics['sentence_bleu2']:.4f}", flush=True)
print(f"Sentence BLEU-3                | {benchmark_metrics['sentence_bleu3']:.4f}", flush=True)
print(f"Sentence BLEU-4                | {benchmark_metrics['sentence_bleu4']:.4f}", flush=True)
print(f"Corpus BLEU-1 (NIST)           | {benchmark_metrics['corpus_bleu1']:.4f}", flush=True)
print(f"Corpus BLEU-2 (NIST)           | {benchmark_metrics['corpus_bleu2']:.4f}", flush=True)
print(f"Corpus BLEU-3 (NIST)           | {benchmark_metrics['corpus_bleu3']:.4f}", flush=True)
print(f"Corpus BLEU-4 (NIST)           | {benchmark_metrics['corpus_bleu4']:.4f}", flush=True)
print(f"ROUGE-1                        | {benchmark_metrics['rouge1']:.4f}", flush=True)
print(f"ROUGE-2                        | {benchmark_metrics['rouge2']:.4f}", flush=True)
print(f"ROUGE-L                        | {benchmark_metrics['rougeL']:.4f}", flush=True)
print(f"Distinct-2                     | {benchmark_metrics['distinct_2']:.4f}", flush=True)
print(f"Clinical_Macro_F1_All14        | {benchmark_metrics['clinical_macro_f1_all14']:.4f}", flush=True)
print(f"Clinical_Macro_F1_Core10       | {benchmark_metrics['clinical_macro_f1_core10']:.4f}", flush=True)
print(f"Clinical_Micro_F1              | {benchmark_metrics['clinical_micro_f1']:.4f}", flush=True)
print(f"Clinical_Micro_Precision       | {benchmark_metrics['clinical_micro_precision']:.4f}", flush=True)
print(f"Clinical_Micro_Recall          | {benchmark_metrics['clinical_micro_recall']:.4f}", flush=True)
print("="*80, flush=True)

print("\n" + "="*110, flush=True)
print("PER-PATHOLOGY CLINICAL BREAKDOWN: Model 4 Qwen-3B (Proposal N >= 150 Threshold Flagging)", flush=True)
print("="*110, flush=True)
for r in pathology_rows:
    print(f"{r['pathology']:<26} | {r['true_positives']:<5} | {r['false_positives']:<5} | {r['false_negatives']:<5} | {r['true_negatives']:<5} | {r['precision']:.4f}  | {r['recall']:.4f}  | {r['f1_score']:.4f}  | [{r['sample_flag']}]", flush=True)

# -------------------------------------------------------------------------
# Plot & Save Pathology Confusion Heatmaps
# -------------------------------------------------------------------------
fig, axes = plt.subplots(4, 4, figsize=(18, 16))
axes = axes.flatten()

for idx, col in enumerate(CHEXPERT_COLS):
    cm = confusion_matrix(y_true[:, idx], y_pred[:, idx], labels=[1, 0])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[idx],
                xticklabels=["Pos", "Neg"], yticklabels=["Pos", "Neg"])
    f1_val = pathology_rows[idx]['f1_score']
    axes[idx].set_title(f"{col}\n(F1: {f1_val:.3f})", fontsize=11, fontweight="bold")
    axes[idx].set_ylabel("True")
    axes[idx].set_xlabel("Pred")

for idx in range(len(CHEXPERT_COLS), len(axes)):
    fig.delaxes(axes[idx])

plt.suptitle("Clinical Pathology Confusion Matrix Breakdown (Test Set N=6388)", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("/kaggle/working/pathology_confusion_matrices.png", dpi=300, bbox_inches="tight")
plt.close()
print("Confusion Matrices successfully saved to /kaggle/working/pathology_confusion_matrices.png", flush=True)

print(f"\nEvaluation completed in {(time.time()-eval_start_time)/60:.1f} minutes.", flush=True)
print("All Model 4 test artifacts, heatmaps and JSON/CSV reports saved successfully to /kaggle/working/", flush=True)



Evaluating Model 4 (RAD-DINO + Qwen-2.5-3B-Instruct) on: cuda


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Model 4 Checkpoint (Epoch 2) loaded successfully with verified PEFT LoRA state.


Generating clinical reports on 6388 test cases using Qwen-3B with True 4D Prefix Attention...


Evaluating Model 4 Test Split:   0%|          | 0/6388 [00:00<?, ?it/s]